<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


<h1>实验：卷积神经网络</h1> 



<h3>本笔记本目标</h3>    
<h5> 1. 学习如何使用卷积神经网络对 MNIST 数据库中的手写数字进行分类</h5>
<h5> 2. 学习如何重塑图像以加快处理速度 </h5>     



<h2>目录</h2>
<p>在本实验中，我们将使用卷积神经网络对 MNIST 数据库中的手写数字进行分类。我们将重塑图像以加快处理速度。</p>

- [获取数据](#Get-the-Data)
- [构建卷积神经网络类](#Build-a-Convolutional-Neural-Network-Class)
- [定义卷积神经网络分类器、损失函数、优化器并训练模型](#Define-the-Convolutional-Neural-Network-Classifier,-Criterion-function,-Optimizer,-and-Train-the-Model)
- [分析结果](#Analyze-Results)

<p>预计所需时间：<strong>25 分钟</strong> </p>

<hr>


<h2>准备工作</h2>


In [ ]:
%%time
%pip install numpy matplotlib
%pip install torch==2.8.0+cpu torchvision==0.23.0+cpu torchaudio==2.8.0+cpu \
--index-url https://download.pytorch.org/whl/cpu


In [ ]:
# 导入本实验需要使用的库

# 使用以下代码安装 torchvision 库
# !conda install -y torchvision

# PyTorch 库
import torch
# PyTorch Neural Network
import torch.nn as nn
# 允许我们转换数据
import torchvision.transforms as transforms
# 允许我们下载数据集
import torchvision.datasets as dsets
# 用于绘制数据和损失曲线
import matplotlib.pylab as plt
# 允许我们使用数组来操作和存储数据
import numpy as np


定义 <code>plot_channels</code> 函数来绘制每个通道的卷积核参数


In [ ]:
# 定义绘制通道的函数

def plot_channels(W):
    n_out = W.shape[0]
    n_in = W.shape[1]
    w_min = W.min().item()
    w_max = W.max().item()
    fig, axes = plt.subplots(n_out, n_in)
    fig.subplots_adjust(hspace=0.1)
    out_index = 0
    in_index = 0
    
    # 将输出作为行绘制，输入作为列 
    for ax in axes.flat:
        if in_index > n_in-1:
            out_index = out_index + 1
            in_index = 0
        ax.imshow(W[out_index, in_index, :, :], vmin=w_min, vmax=w_max, cmap='seismic')
        ax.set_yticklabels([])
        ax.set_xticklabels([])
        in_index = in_index + 1

    plt.show()


定义 <code>plot_parameters</code> 函数来绘制具有多个输出的每个通道的卷积核参数。


In [ ]:
# 定义绘制参数的函数

def plot_parameters(W, number_rows=1, name="", i=0):
    W = W.data[:, i, :, :]
    n_filters = W.shape[0]
    w_min = W.min().item()
    w_max = W.max().item()
    fig, axes = plt.subplots(number_rows, n_filters // number_rows)
    fig.subplots_adjust(hspace=0.4)

    for i, ax in enumerate(axes.flat):
        if i < n_filters:
            # 设置子图的标签。
            ax.set_xlabel("kernel:{0}".format(i + 1))

            # 绘制图像。
            ax.imshow(W[i, :], vmin=w_min, vmax=w_max, cmap='seismic')
            ax.set_xticks([])
            ax.set_yticks([])
    plt.suptitle(name, fontsize=10)    
    plt.show()


定义 <code>plot_activation</code> 函数来绘制卷积层的激活图


In [ ]:
# 定义绘制激活图的函数

def plot_activations(A, number_rows=1, name="", i=0):
    A = A[0, :, :, :].detach().numpy()
    n_activations = A.shape[0]
    A_min = A.min().item()
    A_max = A.max().item()
    fig, axes = plt.subplots(number_rows, n_activations // number_rows)
    fig.subplots_adjust(hspace = 0.9)    

    for i, ax in enumerate(axes.flat):
        if i < n_activations:
            # 设置子图的标签。
            ax.set_xlabel("activation:{0}".format(i + 1))

            # 绘制图像。
            ax.imshow(A[i, :], vmin=A_min, vmax=A_max, cmap='seismic')
            ax.set_xticks([])
            ax.set_yticks([])
    plt.show()


定义 <code>show_data</code> 函数将数据样本绘制为图像。


In [ ]:
def show_data(data_sample):
    plt.imshow(data_sample[0].numpy().reshape(IMAGE_SIZE, IMAGE_SIZE), cmap='gray')
    plt.title('y = '+ str(data_sample[1]))


<!--Empty Space for separating topics-->


<h2 id="Makeup_Data">获取数据</h2> 


我们创建一个变换来调整图像大小并将其转换为张量。


In [ ]:
IMAGE_SIZE = 16

# 首先调整图像大小，然后将其转换为张量
composed = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.ToTensor()])


通过将参数 <code>train</code> 设置为 <code>True</code> 来加载训练数据集。我们使用上面定义的变换。


In [ ]:
train_dataset = dsets.MNIST(root='./data', train=True, download=True, transform=composed)


通过将参数 train 设置为 <code>False</code> 来加载测试数据集。


In [ ]:
validation_dataset = dsets.MNIST(root='./data', train=False, download=True, transform=composed)


矩形张量中的每个元素对应一个表示像素强度的数字，如下图所示。


<img src="https://s3-api.us-geo.objectstorage.softlayer.net/cf-courses-data/CognitiveClass/DL0110EN/notebook_images%20/chapter%206/6.2.1imagenet.png" width="550" alt="MNIST data image">


输出第四个标签


In [ ]:
# 第四个数据元素的标签

train_dataset[3][1]


绘制第四个样本


In [ ]:
# 第四个数据元素的图像
show_data(train_dataset[3])


第四个样本是 "1"。


<!--Empty Space for separating topics-->


<h2 id="CNN">构建卷积神经网络类</h2>


构建一个具有两个卷积层和一个全连接层的卷积网络类。预先确定最终输出矩阵的大小。构造函数中的参数是第一层和第二层的输出通道数。

通道宽度可以使用以下公式计算

![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-CV0101EN-Coursera/labs/Module4/Channel_Width.png)

每个 CNN 层和最大池化层之后都必须计算通道宽度。

CNN 层默认值：

步幅：1
填充：0
空洞：1

最大池化层默认值：

步幅：卷积核大小
填充：0
空洞：1


In [ ]:
class CNN(nn.Module):
    
    # 构造函数
    def __init__(self, out_1=16, out_2=32):
        super(CNN, self).__init__()
        # 我们从 1 个通道开始，因为图像是单通道的黑白图像
        # 该层后的通道数量为 16，空间尺寸：(\frac{16 - 5 + 2\times2}{1} + 1 = 16)
        self.cnn1 = nn.Conv2d(in_channels=1, out_channels=out_1, kernel_size=5, padding=2)
        # 该层后的通道数量为 16，空间尺寸：(16 / 2 = 8)
        self.maxpool1=nn.MaxPool2d(kernel_size=2)
        
        # 该层后的通道数量为 32，空间尺寸：(\frac{8 - 5 + 2\times2}{1} + 1 = 8)
        self.cnn2 = nn.Conv2d(in_channels=out_1, out_channels=out_2, kernel_size=5, stride=1, padding=2)
        # 该层后的通道数量为 32，空间尺寸：(8 / 2 = 4)
        self.maxpool2=nn.MaxPool2d(kernel_size=2)
        # 根据上面的宽度计算，我们总共有 out_2 (32) 个通道，每个通道大小为 4×4。通道是正方形的。
        # 输出是每个类别的值，共有10个类别
        self.fc1 = nn.Linear(out_2 * 4 * 4, 10)
    
    # 预测
    def forward(self, x):
        # 将 X 值通过每个 CNN、ReLU 和池化层，然后将其展平以输入全连接层
        x = self.cnn1(x)
        x = torch.relu(x)
        x = self.maxpool1(x)
        x = self.cnn2(x)
        x = torch.relu(x)
        x = self.maxpool2(x)
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        return x
    
    # 输出 CNN、ReLU 和池化层每个阶段的结果
    def activations(self, x):
        # 输出激活图，这不是必需的
        z1 = self.cnn1(x)
        a1 = torch.relu(z1)
        out = self.maxpool1(a1)
        
        z2 = self.cnn2(out)
        a2 = torch.relu(z2)
        out1 = self.maxpool2(a2)
        out = out.view(out.size(0),-1)
        return z1, a1, z2, a2, out1,out


<h2 id="Train">定义卷积神经网络分类器、损失函数、优化器并训练模型</h2> 


第一层有 16 个输出通道，第二层有 32 个输出通道


In [ ]:
# 使用 CNN 类创建模型对象

model = CNN(out_1=16, out_2=32)


在训练卷积核之前绘制卷积核的模型参数。卷积核是随机初始化的。


In [ ]:
# 绘制参数

plot_parameters(model.state_dict()['cnn1.weight'], number_rows=4, name="1st layer kernels before training ")
plot_parameters(model.state_dict()['cnn2.weight'], number_rows=4, name='2nd layer kernels before training' )


定义损失函数、优化器和数据加载器


In [ ]:
# 创建一个用于测量损失的准则
criterion = nn.CrossEntropyLoss()
learning_rate = 0.1
# 创建一个使用学习率和梯度更新模型参数的优化器
optimizer = torch.optim.SGD(model.parameters(), lr = learning_rate)
# 为训练数据创建一个批量大小为 100 的数据加载器 
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=100)
# 为验证数据创建一个批量大小为 5000 的数据加载器 
validation_loader = torch.utils.data.DataLoader(dataset=validation_dataset, batch_size=5000)


训练模型并确定验证准确率（技术上为测试准确率）**（这可能需要很长时间）**


In [ ]:
# 训练模型

# 我们希望在训练数据集上训练的次数
n_epochs=3
# 用于跟踪代价和准确率的列表
cost_list=[]
accuracy_list=[]
# 验证数据集的大小
N_test=len(validation_dataset)

# 模型训练函数
def train_model(n_epochs):
    # 每个轮次循环
    for epoch in range(n_epochs):
        # 跟踪每个轮次的代价
        COST=0
        # 遍历训练加载器中的每个批次
        for x, y in train_loader:
            # 重置计算得到的梯度值，每次都必须这样做，因为如果不重置，梯度会累积
            optimizer.zero_grad()
            # 基于 X 值进行预测
            z = model(x)
            # 测量预测值与实际 Y 值之间的损失
            loss = criterion(z, y)
            # 计算每个权重和偏置的梯度值
            loss.backward()
            # 根据计算得到的梯度值更新权重和偏置
            optimizer.step()
            # 累积损失 
            COST+=loss.data
        
        # 保存每个轮次训练数据的代价
        cost_list.append(COST)
        # 跟踪正确预测的数量
        correct=0
        # 对验证数据进行预测  
        for x_test, y_test in validation_loader:
            # 进行预测
            z = model(x_test)
            # 最大值对应的类别就是我们的预测结果
            _, yhat = torch.max(z.data, 1)
            # 检查预测是否与实际值匹配
            correct += (yhat == y_test).sum().item()
        
        # 计算准确率并保存
        accuracy = correct / N_test
        accuracy_list.append(accuracy)
     
train_model(n_epochs)


<!--Empty Space for separating topics-->


<h2 id="Result">分析结果</h2> 


绘制验证数据上的损失和准确率：


In [ ]:
# 绘制损失和准确率随轮次变化的图

fig, ax1 = plt.subplots()
color = 'tab:red'
ax1.plot(cost_list, color=color)
ax1.set_xlabel('epoch', color=color)
ax1.set_ylabel('Cost', color=color)
ax1.tick_params(axis='y', color=color)
    
ax2 = ax1.twinx()  
color = 'tab:blue'
ax2.set_ylabel('accuracy', color=color) 
ax2.set_xlabel('epoch', color=color)
ax2.plot( accuracy_list, color=color)
ax2.tick_params(axis='y', color=color)
fig.tight_layout()


查看卷积层参数的结果


In [ ]:
# 绘制通道

plot_channels(model.state_dict()['cnn1.weight'])
plot_channels(model.state_dict()['cnn2.weight'])


考虑以下样本


In [ ]:
# 显示第二张图像

show_data(train_dataset[1])


确定激活图


In [ ]:
# 使用 CNN 激活类查看各阶段结果

out = model.activations(train_dataset[1][0].view(1, 1, IMAGE_SIZE, IMAGE_SIZE))


绘制第一组激活图


In [ ]:
# 绘制第一个 CNN 后的输出

plot_activations(out[0], number_rows=4, name="Output after the 1st CNN")


下图是应用 ReLU 激活函数后的结果


In [ ]:
# 绘制第一个 Relu 后的输出

plot_activations(out[1], number_rows=4, name="Output after the 1st Relu")


下图是第二个输出层后的激活图结果。


In [ ]:
# 绘制第二个 CNN 后的输出

plot_activations(out[2], number_rows=32 // 4, name="Output after the 2nd CNN")


下图是应用第二个 ReLU 后的激活图结果


In [ ]:
# 绘制第二个 Relu 后的输出

plot_activations(out[3], number_rows=4, name="Output after the 2nd Relu")


我们可以看到第三个样本的结果


In [ ]:
# 显示第三张图像

show_data(train_dataset[2])


In [ ]:
# 使用 CNN 激活类查看各阶段结果

out = model.activations(train_dataset[2][0].view(1, 1, IMAGE_SIZE, IMAGE_SIZE))


In [ ]:
# 绘制第一个 CNN 后的输出

plot_activations(out[0], number_rows=4, name="Output after the 1st CNN")


In [ ]:
# 绘制第一个 Relu 后的输出

plot_activations(out[1], number_rows=4, name="Output after the 1st Relu")


In [ ]:
# 绘制第二个 CNN 后的输出

plot_activations(out[2], number_rows=32 // 4, name="Output after the 2nd CNN")


In [ ]:
# 绘制第二个 Relu 后的输出

plot_activations(out[3], number_rows=4, name="Output after the 2nd Relu")


绘制前五个分类错误的样本：


In [ ]:
# 绘制分类错误的样本

count = 0
for x, y in torch.utils.data.DataLoader(dataset=validation_dataset, batch_size=1):
    z = model(x)
    _, yhat = torch.max(z, 1)
    if yhat != y:
        show_data((x, y))
        plt.show()
        print("yhat: ",yhat)
        count += 1
    if count >= 5:
        break  


<h2>关于作者：</h2> 

<a href="https://www.linkedin.com/in/joseph-s-50398b136/">Joseph Santarcangelo</a> 拥有电气工程博士学位，他的研究重点是利用机器学习、信号处理和计算机视觉来确定视频如何影响人类认知。Joseph 自获得博士学位以来一直在 IBM 工作。


其他贡献者：<a href="https://www.linkedin.com/in/michelleccarey/">Michelle Carey</a>、<a href="https://www.linkedin.com/in/jiahui-mavis-zhou-a4537814a">Mavis Zhou</a>


感谢 Magnus <a href="http://www.hvass-labs.org/">Erik Hvass Pedersen</a>，他的教程帮助我理解卷积神经网络。



<!--## Change Log

|  Date (YYYY-MM-DD) |  Version | Changed By  |  Change Description |
|---|---|---|---|
| 2025-07-11  | 2.0  | Sathya  |  Converted the lab to Jupyterlab Current |
| 2020-09-23  | 2.0  | Srishti  |  Migrated Lab to Markdown and added to course repo in GitLab |-->



<hr>

## <h3 align="center"> © IBM Corporation. All rights reserved. <h3/>
